# Lab 21 - Final FULL run (Option B)

Chọn T4 GPU. Chạy tuần tự: setup, NB1/p95 + FULL pipeline, report + verify, upload Hugging Face. Không đặt EVAL_LIMIT. Verdict FAIL vẫn là kết quả hợp lệ nếu được phân tích trung thực. Upload dùng token Write hiện có qua Colab Secrets HF_TOKEN; không gửi token trong chat hoặc commit lên GitHub.

In [ ]:
from pathlib import Path
import os,sys,subprocess
ROOT=Path('/content/Day21-Track3-Finetuning-Lab')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/AnhQun18/Day21-Track3-Finetuning-Lab.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)


## Pipeline cuối
Mask assistant-only, epochs=2, MAX_LENGTH từ token_stats. NB2 FULL đóng băng trước NB3; NB4 cùng ngân sách steps. FORCE_RETRAIN bảo đảm các adapter thuộc cùng lượt.

In [ ]:
import os,sys,json,subprocess
from pathlib import Path
from datetime import datetime,timezone
os.environ.update(COMPUTE_TIER='T4',MASK_MODE='assistant-only',EPOCHS='2',FORCE_RETRAIN='1')
os.environ.pop('EVAL_LIMIT',None); os.environ.pop('ONLY',None); os.environ.pop('MAX_LENGTH',None)
subprocess.run([sys.executable,'-m','pytest','tests/','--tb=short','-rN'],check=True)
subprocess.run([sys.executable,'-u','scripts/colab_run.py','nb1'],check=True)
stats=json.loads(Path('results/token_stats.json').read_text())
os.environ['MAX_LENGTH']=str(stats['suggested_max_length'])
print('p95:',stats['p95'],'chosen max_length:',os.environ['MAX_LENGTH'])
if Path('results/runs.csv').exists():
    archive=Path('/content/lab21_prior_runs'); archive.mkdir(exist_ok=True)
    Path('results/runs.csv').rename(archive/('runs_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.csv'))
with Path('/content/lab21_final_run.log').open('w') as log:
    proc=subprocess.Popen([sys.executable,'-u','scripts/colab_run.py','nb1','nb2','nb3','nb4','nb5'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end=''); log.write(line); log.flush()
    if proc.wait(): raise RuntimeError('Pipeline failed: inspect final_run.log')


## Report và gatekeeper
Lấy số trực tiếp từ results; kiểm tra bài nộp độc lập với verdict của model.

In [ ]:
import subprocess,sys,os
subprocess.run([sys.executable,'scripts/build_submission.py'],check=True)
env={k:v for k,v in os.environ.items() if k!='MAX_LENGTH'}
subprocess.run([sys.executable,'scripts/verify.py'],check=True,env=env)


## Option B: adapter trên Hugging Face
Chỉ chạy sau khi report/results hoàn tất. Repo công khai để giảng viên tải adapter; GitHub chứa report, results, LINKS.md và source. Báo cáo: Trần Anh Quân, mã học viên 2A202602598.

In [ ]:
import os, sys, subprocess
from google.colab import userdata
from huggingface_hub import HfApi
token=userdata.get('HF_TOKEN')
repo_id=HfApi(token=token).whoami()['name']+'/Lab21-LoRA-CSKH'
os.environ['LAB21_HF_REPO']=repo_id
try:
    os.environ['HF_TOKEN']=token
    subprocess.run([sys.executable,'scripts/upload_submission_hf.py',repo_id],check=True)
finally:
    os.environ.pop('HF_TOKEN',None)
    del token
